In [ ]:
# Figure 2, Panel A — supplied CODEX feature-extraction and clustering workflow
#
# The current final panel is supplied source artwork. This cell reuses that
# exact PNG when it is available locally and verifies its recorded hash; it
# never redraws or fabricates the workflow.
from hashlib import sha256
import os
from pathlib import Path
import shutil
from IPython.display import Image, display

repository_root = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / 'src' / 'llm_spatial_omics_clustering').is_dir()
)
configured_source = os.environ.get('FIGURE_02_PANEL_A_SOURCE')
source_candidates = [Path(configured_source).expanduser()] if configured_source else []
source_candidates.extend(
    candidate / 'outputs' / 'annotation_accuracy_maximization' /
    'historical_v1_panel_replays' / 'figure_02' / 'panel_a' /
    'figure_02a_v1_supplied_flowchart.png'
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
)
source_path = next((path for path in source_candidates if path.is_file()), None)
if source_path is None:
    raise FileNotFoundError(
        'VERIFY: Figure 2 Panel A source artwork is unavailable; set FIGURE_02_PANEL_A_SOURCE.'
    )
expected_sha256 = 'e5223328c7b5dc2b256e30f71a2bbf3faade138f95e264c5fb44da0af83f10c6'
actual_sha256 = sha256(source_path.read_bytes()).hexdigest()
if actual_sha256 != expected_sha256:
    raise ValueError(f'Figure 2 Panel A source hash mismatch: {actual_sha256}')
output_path = repository_root / 'outputs' / 'figure_02' / 'figure_02a_workflow_source.png'
output_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source_path, output_path)
display(Image(filename=str(output_path)))
print(f'Source: {source_path}')
print(f'Panel PNG: {output_path.relative_to(repository_root)}')


In [ ]:
# Panel B — Ground-truth cell type distribution
# This is the sole executable cell for Figure 2 Panel B. It reads the declared
# B004 subset directly from the source H5AD, validates the raw-label count
# contract, and writes the count table plus manuscript PNG/PDF outputs.
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
repository_root = next(
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / "src" / "llm_spatial_omics_clustering").is_dir()
)
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))

from llm_spatial_omics_clustering.figure_02 import (
    load_color_map,
    load_figure_config,
    load_panel_b_distribution,
    render_panel_b,
    resolve_data_root,
    save_panel_b_provenance,
)

config = load_figure_config(repository_root / "configs" / "figure_02.yaml")
data_root = resolve_data_root(config, panel_key="panel_b")
panel_b_distribution = load_panel_b_distribution(config, data_root=data_root)
output_config = config["panel_b"]["outputs"]

count_path = repository_root / output_config["counts_csv"]
count_path.parent.mkdir(parents=True, exist_ok=True)
panel_b_distribution.counts.to_csv(count_path, index=False)

color_map = load_color_map(
    panel_b_distribution.counts["Cell Type"],
    config,
    data_root=data_root,
    panel_key="panel_b",
)
panel_png = render_panel_b(
    panel_b_distribution.counts,
    repository_root / output_config["panel_png"],
    color_map=color_map,
    style=config["panel_b"]["style"],
)
panel_pdf = render_panel_b(
    panel_b_distribution.counts,
    repository_root / output_config["panel_pdf"],
    color_map=color_map,
    style=config["panel_b"]["style"],
)
provenance_path = save_panel_b_provenance(
    panel_b_distribution,
    repository_root / output_config["provenance_json"],
    config,
)

print(f"B004 source cells: {panel_b_distribution.source_cell_count:,}")
print(f"Plotted cells: {int(panel_b_distribution.counts['Cell Count'].sum()):,}")
print(f"Cell types: {len(panel_b_distribution.counts)}")
print(f"Excluded labels: {panel_b_distribution.excluded_counts}")
print(f"Count table: {count_path.relative_to(repository_root)}")
print(f"Panel PNG: {panel_png.relative_to(repository_root)}")
print(f"Panel PDF: {panel_pdf.relative_to(repository_root)}")
print(f"Provenance: {provenance_path.relative_to(repository_root)}")


In [ ]:
# Panel C — Ground-truth tissue region
# This is the sole executable cell for Figure 2 Panel C. It reads one verified
# B004 tissue region directly from the source H5AD, validates its exact keys,
# labels, and coordinate bounds, then renders the supplied spatial reference.
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
repository_root = next(
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / "src" / "llm_spatial_omics_clustering").is_dir()
)
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))

from llm_spatial_omics_clustering.figure_02 import (
    load_color_map,
    load_figure_config,
    load_panel_c_spatial_data,
    render_panel_c,
    resolve_data_root,
    save_panel_c_provenance,
)

# The config records the screenshot-matched B004-A-404 File_ID, raw-label
# contract, native x/y orientation, and the presentation settings below.
config = load_figure_config(repository_root / "configs" / "figure_02.yaml")
data_root = resolve_data_root(config, panel_key="panel_c")
panel_c_data = load_panel_c_spatial_data(config, data_root=data_root)
panel_c_config = config["panel_c"]
output_config = panel_c_config["outputs"]
truth_column = panel_c_config["labels"]["truth_column"]
coordinate_columns = panel_c_config["coordinates"]["columns"]

# Save the exact H5AD subset used in the plot as a local, gitignored audit
# table. This is convenient for reviewing the plotted cells without reading
# the full H5AD again.
cells_path = repository_root / output_config["cells_csv"]
cells_path.parent.mkdir(parents=True, exist_ok=True)
panel_c_data.cells.to_csv(cells_path, index=False)

# The tracked color key fixes the raw cell-type palette; Noise is intentionally
# retained here because its pale-pink cells are visible in the supplied panel.
color_map = load_color_map(
    panel_c_data.cells[truth_column],
    config,
    data_root=data_root,
    panel_key="panel_c",
)
style = panel_c_config["style"]
panel_png = render_panel_c(
    panel_c_data.cells,
    repository_root / output_config["panel_png"],
    label_column=truth_column,
    coordinate_columns=coordinate_columns,
    color_map=color_map,
    style=style,
)
panel_pdf = render_panel_c(
    panel_c_data.cells,
    repository_root / output_config["panel_pdf"],
    label_column=truth_column,
    coordinate_columns=coordinate_columns,
    color_map=color_map,
    style=style,
)
provenance_path = save_panel_c_provenance(
    panel_c_data,
    repository_root / output_config["provenance_json"],
    config,
)

region = panel_c_config["region"]
print(f"Region: {region['source_slide_name']} ({region['file_id']})")
print(f"Plotted raw-label cells: {len(panel_c_data.cells):,}")
print(f"Raw cell types: {len(panel_c_data.cell_type_counts)}")
print(f"Noise cells retained: {panel_c_data.cell_type_counts['Noise']:,}")
print(f"Cell table: {cells_path.relative_to(repository_root)}")
print(f"Panel PNG: {panel_png.relative_to(repository_root)}")
print(f"Panel PDF: {panel_pdf.relative_to(repository_root)}")
print(f"Provenance: {provenance_path.relative_to(repository_root)}")


In [ ]:
# Panel D — Ground Truth and four method-derived UMAP views
# This is the sole executable cell for Figure 2 Panel D. It loads only the
# declared B004 cohort from the source H5AD, then validates each keyed method
# assignment before rendering the five-view panel.
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
repository_root = next(
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / "src" / "llm_spatial_omics_clustering").is_dir()
)
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))

from llm_spatial_omics_clustering.figure_02 import (
    build_panel_d_table,
    build_shared_umap,
    load_b004_h5ad,
    load_color_map,
    load_figure_config,
    load_method_assignments,
    render_panel_d,
    resolve_data_root,
    save_panel_d_provenance,
)

config = load_figure_config(repository_root / "configs" / "figure_02.yaml")
data_root = resolve_data_root(config)
panel_d_data = load_b004_h5ad(config, data_root=data_root)
panel_d_coordinates = build_shared_umap(panel_d_data, config)
panel_d_assignments = load_method_assignments(panel_d_data, config, data_root=data_root)
panel_d_table = build_panel_d_table(
    panel_d_data,
    panel_d_coordinates,
    panel_d_assignments,
    config,
)

output_config = config["panel_d"]["outputs"]
coordinate_path = repository_root / output_config["coordinates_csv"]
coordinate_path.parent.mkdir(parents=True, exist_ok=True)
panel_d_coordinates.to_csv(coordinate_path, index=False)

style = config["panel_d"]["style"]
color_map = load_color_map(
    panel_d_table["Ground Truth"],
    config,
    data_root=data_root,
)
panel_png = render_panel_d(
    panel_d_table,
    repository_root / output_config["panel_png"],
    color_map=color_map,
    zoom=style["zoom"],
    point_size=float(style["point_size"]),
    alpha=float(style["point_alpha"]),
)
panel_pdf = render_panel_d(
    panel_d_table,
    repository_root / output_config["panel_pdf"],
    color_map=color_map,
    zoom=style["zoom"],
    point_size=float(style["point_size"]),
    alpha=float(style["point_alpha"]),
)
provenance_path = save_panel_d_provenance(
    panel_d_data,
    panel_d_table,
    repository_root / output_config["provenance_json"],
    config,
)

print(f"B004 cells: {len(panel_d_table):,}")
print("Clusters:", {name: int(table['cluster'].nunique()) for name, table in panel_d_assignments.items()})
print(f"Coordinates: {coordinate_path.relative_to(repository_root)}")
print(f"Panel PNG: {panel_png.relative_to(repository_root)}")
print(f"Panel PDF: {panel_pdf.relative_to(repository_root)}")
print(f"Provenance: {provenance_path.relative_to(repository_root)}")


In [ ]:
# Panel E — Cell-type classification weighted F1 by clustering method
# This is the sole executable cell for Figure 2 Panel E. It reads B004 reference
# labels from the source H5AD, applies the documented 20-class evaluation map,
# calculates a region-specific cluster-majority mapping, and plots all eight
# B004 regional observations. PIXIE deliberately uses the validated 50-cluster
# image-native TIFF assignment configured for Panel D.
from pathlib import Path
import sys

# Locate the repository from any Jupyter working directory, then make the
# package importable without relying on a manually activated editable install.
working_directory = Path.cwd().resolve()
repository_root = next(
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / "src" / "llm_spatial_omics_clustering").is_dir()
)
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))

from llm_spatial_omics_clustering.figure_02 import (
    load_figure_config,
    load_panel_e_metrics,
    render_panel_e,
    resolve_data_root,
    save_panel_e_provenance,
)

# The tracked configuration fixes the B004 File_ID cohort, the truth-label
# harmonization, exact method-assignment artifacts, visual styling, and
# expected regional scores. It rejects a stale or differently parameterized
# TIFF PIXIE result rather than silently plotting it.
config = load_figure_config(repository_root / "configs" / "figure_02.yaml")
data_root = resolve_data_root(config, panel_key="panel_e")
panel_e_data = load_panel_e_metrics(config, data_root=data_root)
panel_e = config["panel_e"]

# Save the machine-readable 8 regions × 4 methods source table before rendering
# so the figure's dots, box summaries, and exact per-region values are auditable.
outputs = {
    name: repository_root / relative_path
    for name, relative_path in panel_e["outputs"].items()
}
outputs["scores_csv"].parent.mkdir(parents=True, exist_ok=True)
panel_e_data.scores.to_csv(outputs["scores_csv"], index=False)

# The renderer uses deterministic region colors and point jitter, along with
# standard 1.5-IQR boxplot whiskers. No other Figure 2 panel is executed here.
method_order = [str(method["label"]) for method in panel_e["methods"]]
region_order = [str(file_id) for file_id in panel_e["cohort"]["file_ids"]]
render_panel_e(
    panel_e_data.scores,
    outputs["panel_png"],
    method_order=method_order,
    region_order=region_order,
    style=panel_e["style"],
)
render_panel_e(
    panel_e_data.scores,
    outputs["panel_pdf"],
    method_order=method_order,
    region_order=region_order,
    style=panel_e["style"],
)
save_panel_e_provenance(
    panel_e_data,
    outputs["provenance_json"],
    config,
    data_root=data_root,
)

print(f"B004 source cells: {panel_e_data.source_cell_count:,}")
print(f"Non-Noise evaluation cells: {panel_e_data.evaluation_cell_count:,}")
print(f"Harmonized evaluation classes: {panel_e_data.evaluation_class_count}")
print(f"Cluster counts: {panel_e_data.cluster_counts}")
for method, score in panel_e_data.scores.groupby("method", sort=False)["weighted_f1"].mean().items():
    print(f"{method} mean regional weighted F1: {score:.6f}")
for artifact_name, artifact_path in outputs.items():
    print(f"{artifact_name}: {artifact_path.relative_to(repository_root)}")


In [ ]:
# Figure 2, Panel F — F1 score by reference cell type
#
# This cell is self-contained. It uses B004 H5AD reference labels, excludes
# Noise, harmonizes raw labels to the documented 20-class evaluation set, and
# maps every method's clusters to a global majority reference label. PIXIE uses
# the image-native 50-cluster TIFF artifact selected for Panel D, not the
# older table-level MiniSom result used in the supplied legacy screenshot.
from pathlib import Path
import sys

# Locate the repository from any Jupyter working directory, then make the
# package importable without relying on a manually activated editable install.
working_directory = Path.cwd().resolve()
repository_root = next(
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / "src" / "llm_spatial_omics_clustering").is_dir()
)
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))

from llm_spatial_omics_clustering.figure_02 import (
    load_figure_config,
    load_panel_fh_metrics,
    render_panel_cell_type_heatmap,
    resolve_data_root,
    save_panel_cell_type_metrics_provenance,
)

config = load_figure_config(repository_root / "configs" / "figure_02.yaml")
panel_key = "panel_f"
panel = config[panel_key]
data_root = resolve_data_root(config, panel_key=panel_key)
panel_f_data = load_panel_fh_metrics(config, panel_key=panel_key, data_root=data_root)

# Save the source table used by the heatmap and matched cell-count bars.
outputs = {
    name: repository_root / relative_path
    for name, relative_path in panel["outputs"].items()
}
outputs["metrics_csv"].parent.mkdir(parents=True, exist_ok=True)
panel_f_data.metrics.to_csv(outputs["metrics_csv"], index=False)

# The configured row order preserves the supplied reference layout even though
# the TIFF-derived PIXIE column changes its historical all-method sort order.
method_order = [str(method["label"]) for method in panel["methods"]]
for artifact_name in ("panel_png", "panel_pdf"):
    render_panel_cell_type_heatmap(
        panel_f_data.metrics,
        outputs[artifact_name],
        method_order=method_order,
        metric_suffix="f1",
        style=panel["style"],
    )
save_panel_cell_type_metrics_provenance(
    panel_f_data,
    outputs["provenance_json"],
    panel_key=panel_key,
    config=config,
    data_root=data_root,
)

print(f"B004 source cells: {panel_f_data.source_cell_count:,}")
print(f"Non-Noise evaluation cells: {panel_f_data.evaluation_cell_count:,}")
print(f"Cluster counts: {panel_f_data.cluster_counts}")
print(panel_f_data.metrics[["cell_type", *[f"{method}_f1" for method in method_order], "n_cells"]].to_string(index=False))
for artifact_name, artifact_path in outputs.items():
    print(f"{artifact_name}: {artifact_path.relative_to(repository_root)}")


In [ ]:
# Figure 2, Panel G — Regional cluster purity by clustering method
#
# Each point is one B004 FILE_ID. Within each region and method, clusters are
# mapped to their majority 20-class reference label; purity is the fraction of
# non-Noise cells whose mapped cluster label matches that reference label.
# PIXIE comes from the validated image-native TIFF artifact used for Panel D.
from pathlib import Path
import sys

# Locate the repository from any Jupyter working directory, then make the
# package importable without relying on a manually activated editable install.
working_directory = Path.cwd().resolve()
repository_root = next(
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / "src" / "llm_spatial_omics_clustering").is_dir()
)
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))

from llm_spatial_omics_clustering.figure_02 import (
    load_figure_config,
    load_panel_g_metrics,
    render_panel_g,
    resolve_data_root,
    save_panel_g_provenance,
)

config = load_figure_config(repository_root / "configs" / "figure_02.yaml")
panel_key = "panel_g"
panel = config[panel_key]
data_root = resolve_data_root(config, panel_key=panel_key)
panel_g_data = load_panel_g_metrics(config, data_root=data_root)

outputs = {
    name: repository_root / relative_path
    for name, relative_path in panel["outputs"].items()
}
outputs["scores_csv"].parent.mkdir(parents=True, exist_ok=True)
panel_g_data.scores.to_csv(outputs["scores_csv"], index=False)
method_order = [str(method["label"]) for method in panel["methods"]]
region_order = [str(file_id) for file_id in config[panel["cohort_source_panel"]]["cohort"]["file_ids"]]
for artifact_name in ("panel_png", "panel_pdf"):
    render_panel_g(
        panel_g_data.scores,
        outputs[artifact_name],
        method_order=method_order,
        region_order=region_order,
        style=panel["style"],
    )
save_panel_g_provenance(
    panel_g_data,
    outputs["provenance_json"],
    config=config,
    data_root=data_root,
)

print(f"B004 source cells: {panel_g_data.source_cell_count:,}")
print(f"Non-Noise evaluation cells: {panel_g_data.evaluation_cell_count:,}")
print(f"Cluster counts: {panel_g_data.cluster_counts}")
print(panel_g_data.scores.pivot(index="region", columns="method", values="cell_purity").to_string())
for artifact_name, artifact_path in outputs.items():
    print(f"{artifact_name}: {artifact_path.relative_to(repository_root)}")


In [ ]:
# Figure 2, Panel H — Per-cell-type recovery purity by clustering method
#
# The historical panel calls this metric "purity". For each 20-class reference
# cell type, it is the recall of the globally majority-mapped cluster label:
# correctly recovered cells divided by all cells of that reference type. This
# differs from Panel G, whose calculation is made separately per tissue region.
from pathlib import Path
import sys

# Locate the repository from any Jupyter working directory, then make the
# package importable without relying on a manually activated editable install.
working_directory = Path.cwd().resolve()
repository_root = next(
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / "src" / "llm_spatial_omics_clustering").is_dir()
)
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))

from llm_spatial_omics_clustering.figure_02 import (
    load_figure_config,
    load_panel_fh_metrics,
    render_panel_cell_type_heatmap,
    resolve_data_root,
    save_panel_cell_type_metrics_provenance,
)

config = load_figure_config(repository_root / "configs" / "figure_02.yaml")
panel_key = "panel_h"
panel = config[panel_key]
data_root = resolve_data_root(config, panel_key=panel_key)
panel_h_data = load_panel_fh_metrics(config, panel_key=panel_key, data_root=data_root)

outputs = {
    name: repository_root / relative_path
    for name, relative_path in panel["outputs"].items()
}
outputs["metrics_csv"].parent.mkdir(parents=True, exist_ok=True)
panel_h_data.metrics.to_csv(outputs["metrics_csv"], index=False)
method_order = [str(method["label"]) for method in panel["methods"]]
for artifact_name in ("panel_png", "panel_pdf"):
    render_panel_cell_type_heatmap(
        panel_h_data.metrics,
        outputs[artifact_name],
        method_order=method_order,
        metric_suffix="purity",
        style=panel["style"],
    )
save_panel_cell_type_metrics_provenance(
    panel_h_data,
    outputs["provenance_json"],
    panel_key=panel_key,
    config=config,
    data_root=data_root,
)

print(f"B004 source cells: {panel_h_data.source_cell_count:,}")
print(f"Non-Noise evaluation cells: {panel_h_data.evaluation_cell_count:,}")
print(f"Cluster counts: {panel_h_data.cluster_counts}")
print(panel_h_data.metrics[["cell_type", *[f"{method}_purity" for method in method_order], "n_cells"]].to_string(index=False))
for artifact_name, artifact_path in outputs.items():
    print(f"{artifact_name}: {artifact_path.relative_to(repository_root)}")


In [ ]:
# Figure 2, Panel I — CD8+ T-cell protein-marker expression profiles
#
# For each method, the cell group consists of every B004 cell in a globally
# raw-reference-majority CD8+ T cluster. The 45 native H5AD X markers retain
# their stored order. Dot color is a marker-wise min–max scaled group mean;
# dot area is the percentage of selected cells with X > 0. The legacy source
# retained any Noise-reference cells that fall in a CD8+-mapped cluster, and
# this reproducible implementation preserves that behavior (VERIFY: Methods
# do not explicitly state whether those cells should be retained).
from pathlib import Path
import sys

# Locate the repository from any Jupyter working directory, then make the
# package importable without relying on a manually activated editable install.
working_directory = Path.cwd().resolve()
repository_root = next(
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / "src" / "llm_spatial_omics_clustering").is_dir()
)
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))

from llm_spatial_omics_clustering.figure_02 import (
    load_figure_config,
    load_panel_i_data,
    render_panel_i,
    resolve_data_root,
    save_panel_i_provenance,
)

config = load_figure_config(repository_root / "configs" / "figure_02.yaml")
panel_key = "panel_i"
panel = config[panel_key]
data_root = resolve_data_root(config, panel_key=panel_key)
panel_i_data = load_panel_i_data(config, data_root=data_root)

outputs = {
    name: repository_root / relative_path
    for name, relative_path in panel["outputs"].items()
}
outputs["summaries_csv"].parent.mkdir(parents=True, exist_ok=True)
panel_i_data.summaries.to_csv(outputs["summaries_csv"], index=False)
method_order = [str(method["label"]) for method in panel["methods"]]
for artifact_name in ("panel_png", "panel_pdf"):
    render_panel_i(
        panel_i_data.summaries,
        outputs[artifact_name],
        marker_order=panel_i_data.marker_names,
        method_order=method_order,
        style=panel["style"],
    )
save_panel_i_provenance(
    panel_i_data,
    outputs["provenance_json"],
    config=config,
    data_root=data_root,
)

print(f"B004 source cells: {panel_i_data.source_cell_count:,}")
print(f"CD8+-mapped selected cells: {panel_i_data.selected_cell_counts}")
print(f"CD8+-mapped cluster counts: {panel_i_data.selected_cluster_counts}")
print(f"All selected-assignment cluster counts: {panel_i_data.cluster_counts}")
for artifact_name, artifact_path in outputs.items():
    print(f"{artifact_name}: {artifact_path.relative_to(repository_root)}")


In [ ]:
# Figure 2, Panel J — Spatial agreement across clustering methods
#
# Each row shows one whole B004 tissue region and a fixed 900–1,100-cell zoom:
# Ground Truth, Leiden, FlowSOM, SpatialSort, and TIFF PIXIE are colored by
# global raw-reference cluster-majority labels. Reference Noise cells are
# omitted after mapping, while predicted Noise labels remain visible. The low
# and high agreement windows were reselected for TIFF PIXIE; they are not the
# legacy table-PIXIE windows from the supplied screenshot. VERIFY: the local
# candidate-search seed, percentile bounds, and radius grid are implementation
# details not explicitly specified in the attached Methods.
from pathlib import Path
import sys

# Locate the repository from any Jupyter working directory, then make the
# package importable without relying on a manually activated editable install.
working_directory = Path.cwd().resolve()
repository_root = next(
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / "src" / "llm_spatial_omics_clustering").is_dir()
)
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))

from llm_spatial_omics_clustering.figure_02 import (
    load_color_map,
    load_figure_config,
    load_panel_j_data,
    render_panel_j,
    resolve_data_root,
    save_panel_j_provenance,
)

config = load_figure_config(repository_root / "configs" / "figure_02.yaml")
panel_key = "panel_j"
panel = config[panel_key]
data_root = resolve_data_root(config, panel_key=panel_key)
panel_j_data = load_panel_j_data(config, data_root=data_root)

outputs = {
    name: repository_root / relative_path
    for name, relative_path in panel["outputs"].items()
}
outputs["examples_csv"].parent.mkdir(parents=True, exist_ok=True)
panel_j_data.examples.to_csv(outputs["examples_csv"], index=False)
method_order = [str(method["label"]) for method in panel["methods"]]
label_columns = ["Ground Truth", *method_order]
spatial_labels = panel_j_data.cells.loc[:, label_columns].astype(str).to_numpy().ravel().tolist()
color_map = load_color_map(
    spatial_labels,
    config,
    data_root=data_root,
    panel_key=panel_key,
)
for artifact_name in ("panel_png", "panel_pdf"):
    render_panel_j(
        panel_j_data,
        outputs[artifact_name],
        color_map=color_map,
        method_order=method_order,
        coordinate_columns=panel["coordinate_columns"],
        style=panel["style"],
    )
save_panel_j_provenance(
    panel_j_data,
    outputs["provenance_json"],
    config=config,
    data_root=data_root,
)

print(f"B004 source cells: {panel_j_data.source_cell_count:,}")
print(f"Reference-Noise cells excluded from maps: {panel_j_data.excluded_counts}")
print(panel_j_data.examples.to_string(index=False))
for artifact_name, artifact_path in outputs.items():
    print(f"{artifact_name}: {artifact_path.relative_to(repository_root)}")
